In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import os
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from underthesea import word_tokenize
from tqdm import tqdm
import faiss
import pickle

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chuẩn bị data

In [3]:
# SETUP & DATA LOADING

csv_file = '../data/all_recipes_final.csv'
data = pd.read_csv(csv_file)

In [5]:
# Tạo combined text: Title + Description
data['combined_text'] = data['title'] + '. ' + data['description']

print(f"Loaded {len(data)} recipes.")

Loaded 10335 recipes.


In [7]:
data.head()

,title,type_of_food,link,description,ingredients,step,note,num_of_ingredients,cook_time,num_of_people,calories,source,combined_text
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress,Cách muối dưa hành truyền thống. Dưa hành muối...
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress,Su hào xào mực - món cổ Tết Bát Tràng. Đĩa xào...
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress,Canh măng ngày Tết cổ truyền Hà Nội. Măng ngấu...
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress,Giả hạnh nhân - món ngon Tết xưa Hà Nội. Đây l...
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress,"Chả bì ớt xiêm xanh. Chả bì bóng đẹp, gói đều ..."


# Embedding dữ liệu theo 2 cách: tf-idf và sbert

In [8]:
# SENTENCE EMBEDDINGS ( TF-IDF )

stopwords={
    "là", "của", "và", "những", "các", "cho", "với", "trong", "để", "khi",
    "một", "có", "được", "từ", "như", "người", "bạn", "hãy", "sẽ", "đã", "đang",
    "thì", "mà", "bị", "bởi", "cả", "lại", "nên", "này", "kia", "làm", "rằng",
    "về", "do", "bằng", "phải", "tại", "theo", "ra", "vào", "lên", "xuống",
    "đến", "qua", "bởi_vì", "nếu", "nhưng", "tuy_nhiên", "mặc_dù", "hoặc", "hay",
    "rất", "quá", "lắm", "nhiều", "ít", "hơn", "nhất", "khá", "chỉ", "mỗi", "từng",
    "không", "chưa", "chẳng", "đừng", "chớ", "vẫn", "cũng", "thôi", "nhé", "nha","muỗng", "thìa", "gam", "gram", "kg", "kilogam", "lít", "ml", "chén", "bát",
    "tô", "dĩa", "đĩa", "trái", "quả", "củ", "nhánh", "tép", "lát", "khứa", "miếng",
    "ổ", "ổ_bánh_mì", "lon", "hộp", "gói", "bao", "giọt", "nhúm", "nắm",
    "bước", "cách", "làm", "thực_hiện", "chuẩn_bị", "sơ_chế", "chế_biến",
    "hướng_dẫn", "thành_phẩm", "lưu_ý", "mẹo", "bí_quyết", "thưởng_thức",
    "bắt_đầu", "tiếp_theo", "sau_đó", "cuối_cùng", "hoàn_thành",
    "ngon", "vị", "hương_vị", "món", "ăn", "thơm", "hấp_dẫn", "đậm_đà",
    "gia_vị", "nêm", "nếm", "vừa_ăn", "khẩu_vị", "gia_đình", "bếp",
    "khoảng", "độ", "phút", "giờ", "nóng", "lạnh", "nguội"
}

# Viết hàm tiền xử lý dữ liệu
def processing_data(text):
  text = str(text)
  # Chuyển về từ thường
  text = text.lower()
  # Xóa dấu câu, ký tự đặc biệt
  text = re.sub(r'[^\w\s]', ' ', text)
  # Tách từ bằng underthesea
  text = word_tokenize(text, format="text")
  # Xóa khoảng trắng
  text = re.sub(r'\s+', ' ', text).strip()
  # Lọc stopword
  words = text.split()
  valid_words = []
  for word in words:
      # Kiểm tra từ đó (hoặc từ gốc thay thế dấu _) có trong stopword không
      if word not in stopwords and word.replace('_', ' ') not in stopwords:
          valid_words.append(word)
  return ' '.join(valid_words)

# Xử lý dữ liệu
print("Đang tiền xử lý dữ liệu văn bản...")
tqdm.pandas()
data['embed_tf'] = data['combined_text'].progress_apply(processing_data)

# Sử dụng TfidfVectorizer để chuyển cột combined_text của các món ăn về tf-idf
vectorizer = TfidfVectorizer()
overview_matrix = vectorizer.fit_transform(data['embed_tf'])

# Tính toán cosine bằng linear_kernel
cosine_sim = linear_kernel(overview_matrix, overview_matrix)

# Đánh index cho các món ăn bằng pd.Series() và lưu trong biến mapping
mapping = pd.Series(data.index, index=data['title']).drop_duplicates()

print("Đang chuyển đổi sang định dạng FAISS...")
# Chuyển Sparse Matrix -> Dense Matrix -> Float32
tfidf_dense = overview_matrix.toarray().astype('float32')

# Khởi tạo Index FAISS
d = tfidf_dense.shape[1] # Số chiều = Số lượng từ vựng (Vocabulary size)

# Dùng IndexFlatIP
index = faiss.IndexFlatIP(d)

# Thêm dữ liệu vào Index
index.add(tfidf_dense)
print(f"Đã thêm {index.ntotal} món ăn vào FAISS Index.")

#  Lưu FAISS Index (Chứa cấu trúc tìm kiếm) và Mapping (Chứa quan hệ Tên món -> ID)
faiss.write_index(index, "food_tfidf.index")

with open("food_mapping.pkl", "wb") as f:
    pickle.dump(mapping, f)

print("Đã lưu Index và Mapping xuống ổ cứng!")

Đang tiền xử lý dữ liệu văn bản...


100%|██████████| 10335/10335 [00:52<00:00, 195.26it/s]


Đang chuyển đổi sang định dạng FAISS...
Đã thêm 10335 món ăn vào FAISS Index.
Đã lưu Index và Mapping xuống ổ cứng!


In [9]:
#  SENTENCE EMBEDDINGS ( SBERT )

# Load model keepitreal/vietnamese-sbert
print("Loading Embedding Model...")
model = SentenceTransformer('keepitreal/vietnamese-sbert')

# Generate embeddings
print("Encoding dataset...")
embeddings = model.encode(data['combined_text'].tolist(), show_progress_bar=True)

# Chuẩn hóa L2
faiss.normalize_L2(embeddings)

# Khởi tạo Index FAISS
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)

# Thêm vector vào Index
index.add(embeddings)

# Lưu index và mảng Embeddings
faiss.write_index(index, "food_sbert.index")
np.save("food_embeddings.npy", embeddings)
print("Đã lưu xong: 'food_sbert.index' và 'food_embeddings.npy'")

Loading Embedding Model...
Encoding dataset...


Batches:   1%|          | 2/323 [00:25<1:08:30, 12.80s/it]'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 76f8b90b-bc43-4b52-97cb-721ae8c7d948)')' thrown while requesting HEAD https://huggingface.co/keepitreal/vietnamese-sbert/resolve/refs%2Fpr%2F8/model.safetensors
Retrying in 1s [Retry 1/5].
Batches:   2%|▏         | 8/323 [01:32<1:00:58, 11.61s/it]


KeyboardInterrupt: 

# Các hàm đo độ tương đồng

In [55]:
# Các hàm gợi ý của tf-idf
# Load data faiss tf-idf
index_tf_idf = faiss.read_index("food_tfidf.index")
with open("food_mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

def search_recipes_tf_idf(user_query, top_n=50):
    """
    Tìm kiếm món ăn dựa trên query người dùng sử dụng TF-IDF.
    """
    # Tiền xử lý Query
    processed_query = processing_data(user_query)

    # Vector hóa Query
    query_sparse_matrix = vectorizer.transform([processed_query])

    # Chuyển đổi định dạng cho FAISS
    query_vec = query_sparse_matrix.toarray().astype('float32')

    D, I = index_tf_idf.search(query_vec, k=top_n)
    result_indices = I[0]

    # Lọc bỏ các giá trị -1 (nếu FAISS không tìm đủ k kết quả - hiếm gặp nhưng an toàn)
    result_indices = result_indices[result_indices != -1]

    return data.iloc[result_indices].copy()

In [56]:
# Các hàm gợi ý của sbert
# Load Model để mã hóa query mới của user
model = SentenceTransformer('keepitreal/vietnamese-sbert')

# Load FAISS Index
index_faiss = faiss.read_index("food_sbert.index")

# Load Embeddings gốc
embeddings = np.load("food_embeddings.npy")

# Tìm theo query của User - Bước đệm cho Reranking
def search_recipes_sbert(user_query, top_n=50):
    """
    Dùng SBERT để tìm top-N candidates phù hợp nhất với query
    """
    query_vec = model.encode([user_query])
    # Chuẩn hóa query
    faiss.normalize_L2(query_vec)
    # Tìm kiếm
    D, I = index_faiss.search(query_vec, k=top_n)

    # Lấy danh sách index
    result_indices = I[0]

    return data.iloc[result_indices].copy()

# Rerank bằng llm (gemini)

In [ ]:
# LLM-BASED RERANKING (GEMINI)
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


genai.configure(api_key=GOOGLE_API_KEY)

def llm_rerank(user_query, candidate_df, top_n=5):

    # 1. Chuẩn bị dữ liệu Input
    candidates_json = candidate_df[['title', 'description']].to_json(orient='index', force_ascii=False)

    # 2. Cấu hình System Instruction (Chỉ thị cho AI)
    system_instruction = """
    You are a culinary expert assistant specializing in Vietnamese cuisine.
    Your task is to rerank the provided list of recipes based on their relevance to the user's query.

    CRITICAL OUTPUT RULES:
    1. Return ONLY a valid JSON object.
    2. Do not include any explanations, Markdown formatting (like ```json), or extra text.
    3. The JSON must follow this format: {"ranked_indices": [index1, index2, index3...]}
    4. Sort the indices from most relevant to least relevant.
    """

    # 3. Khởi tạo Model
    model = genai.GenerativeModel(
        model_name='gemini-2.0-flash-lite',
        system_instruction=system_instruction,
        generation_config={"response_mime_type": "application/json"} # Ép kiểu trả về là JSON
    )

    # 4. Tạo Prompt
    user_prompt = f"""
    User Query: "{user_query}"

    Candidate Recipes (JSON format with ID as key):
    {candidates_json}

    Please analyze and rank the top {top_n} most relevant recipes.
    """

    try:
        # 5. Gọi Gemini API
        response = model.generate_content(user_prompt)

        content = response.text.strip()

        # 6. Vệ sinh dữ liệu (Gemini đôi khi vẫn thêm markdown dù đã nhắc)
        if content.startswith("```json"):
            content = content.replace("```json", "").replace("```", "")
        elif content.startswith("```"):
            content = content.replace("```", "")

        content = content.strip()

        # 7. Parse kết quả
        match = re.search(r'(\{.*\}|\[.*\])', content, re.DOTALL)

        if match:
            clean_json_str = match.group(0) # Chỉ lấy phần JSON hợp lệ
            result_json = json.loads(clean_json_str)
        else:
            # Fallback nếu không tìm thấy pattern
            # Cố gắng clean thủ công
            content = content.replace("```json", "").replace("```", "").strip()
            result_json = json.loads(content)

        # Xử lý các trường hợp key khác nhau
        if isinstance(result_json, list):
            ranked_indices = result_json
        elif "ranked_indices" in result_json:
            ranked_indices = result_json["ranked_indices"]
        else:
            # Fallback: Lấy value đầu tiên nếu format lạ
            ranked_indices = list(result_json.values())[0]

        # 8. Trả về DataFrame đã sắp xếp
        # Chỉ lấy các index có tồn tại trong candidate_df để tránh lỗi
        valid_indices = [int(idx) for idx in ranked_indices if int(idx) in candidate_df.index]

        # Nếu LLM trả về ít hơn top_n hoặc rỗng, fallback về danh sách gốc
        if not valid_indices:
            return candidate_df.head(top_n)

        return data.loc[valid_indices]

    except Exception as e:
        print(f"Error during Gemini reranking: {e}")
        # Fallback về danh sách gốc từ Semantic Search nếu lỗi
        return candidate_df.head(top_n)

In [58]:
if __name__ == "__main__":
    # Chọn model : tf-idf hoặc Sbert
    using_model= 'tf-idf'

    query = input()
    print(f"\n--- Những món ăn liên quan phù hợp với yêu cầu '{query}' ---")

    # Lấy ứng viên
    if using_model == 'Sbert':
        candidates = search_recipes_sbert(query, top_n=50)
    else:
        candidates = search_recipes_tf_idf(query, top_n=50)
    display(candidates[['title','description', 'link']].head(10))

    # LLM Reranking
    print(f"\n--- Kết quả rerank sau khi dùng LLM (gemini)  ---")
    reranked_results = llm_rerank(query, candidates, top_n=10)
    display(reranked_results[['title', 'description', 'link']])

canh chua

--- Những món ăn liên quan phù hợp với yêu cầu 'canh chua' ---


,title,description,link
799,Canh măng chua cá lóc,"Bông cải xào thịt bò, canh măng chua cá lóc, c...",https://vnexpress.net/hap-dan-tu-nhung-mon-que...
432,Chia sẻ bí quyết nấu canh chua cá thơm ngon ch...,"Canh cá nấu chua hấp dẫn bởi nước canh trong, ...",https://vnexpress.net/doi-song-cooking-canh-ch...
585,Canh chua cá Nam bộ,"Món canh chua cay nấu theo kiểu miền trong, vớ...",https://vnexpress.net/canh-chua-ca-nam-bo-4311...
717,Cách nấu canh trai nấu chua giải nhiệt mùa hè,"Mùa hè, các bà nội trợ thường ưu tiên lựa chọn...",https://vnexpress.net/canh-trai-nau-chua-giai-...
219,Canh rau sắn chua nấu sườn - món ngon Phú Thọ,"Rau sắn chua nhẹ hòa quyện cùng sườn mềm ngọt,...",https://vnexpress.net/canh-rau-san-chua-nau-su...
235,Canh chua cá trê,Món canh chua cá trê giới thiệu với bạn dưới đ...,https://vnexpress.net/canh-chua-ca-tre-4311448...
340,Canh cá giấm mẻ,Canh cá là món ăn dân dã thơm ngon và bổ dưỡng...,https://vnexpress.net/canh-ca-giam-me-4311166....
278,Cá chép om dưa,Món canh dưa chua rất phù hợp cho tiết trời nó...,https://vnexpress.net/ca-chep-om-dua-4311361.html
70,Cách làm mứt cam dẻo đón Tết,"Từng lát cam dẻo ánh vàng trong veo, vị chua c...",https://vnexpress.net/cach-lam-mut-cam-deo-don...
767,Canh chua tôm nõn,Tôm là món hải sản có vị ngọt tự nhiên rất đượ...,https://vnexpress.net/quen-la-voi-tom-4311462....



--- Kết quả rerank sau khi dùng LLM (gemini)  ---


,title,description,link
432,Chia sẻ bí quyết nấu canh chua cá thơm ngon ch...,"Canh cá nấu chua hấp dẫn bởi nước canh trong, ...",https://vnexpress.net/doi-song-cooking-canh-ch...
585,Canh chua cá Nam bộ,"Món canh chua cay nấu theo kiểu miền trong, vớ...",https://vnexpress.net/canh-chua-ca-nam-bo-4311...
799,Canh măng chua cá lóc,"Bông cải xào thịt bò, canh măng chua cá lóc, c...",https://vnexpress.net/hap-dan-tu-nhung-mon-que...
235,Canh chua cá trê,Món canh chua cá trê giới thiệu với bạn dưới đ...,https://vnexpress.net/canh-chua-ca-tre-4311448...
795,Canh khoai sọ rau nhút,Cá trứng nướng xốt me chua ngọt lạ miệng kết h...,https://vnexpress.net/bua-an-am-cung-voi-4-mon...
382,Canh chua đầu cá diêu hồng nấu đậu rồng,Canh chua đầu cá diêu hồng nấu đậu rồng vị chu...,https://vnexpress.net/canh-chua-dau-ca-dieu-ho...
396,Cách làm canh ngao chua nấu dứa - món đưa cơm ...,"Một bát canh thanh mát với thịt ngao mềm, đan ...",https://vnexpress.net/doi-song-cooking-canh-ng...
717,Cách nấu canh trai nấu chua giải nhiệt mùa hè,"Mùa hè, các bà nội trợ thường ưu tiên lựa chọn...",https://vnexpress.net/canh-trai-nau-chua-giai-...
767,Canh chua tôm nõn,Tôm là món hải sản có vị ngọt tự nhiên rất đượ...,https://vnexpress.net/quen-la-voi-tom-4311462....
425,Cách làm Canh cá tràu nấu khế giải nhiệt mùa hè,"Canh cá tràu (cá quả, cá chuối) nấu khế là món...",https://vnexpress.net/canh-ca-trau-nau-khe-gia...
